In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel
from langchain_core.tracers.context import collect_runs
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
api_key=str(os.getenv("API_KEY"))
base_url=str(os.getenv("BASE_URL"))

In [ ]:
llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    base_url=base_url,
    temperature=0.7,
)

## Chaining invocations

In [ ]:
prompt = PromptTemplate(
    template="Tell me a joke about {topic}"
)

In [ ]:
parser = StrOutputParser()

In [ ]:
parser.invoke(
    llm.invoke(
        prompt.invoke(
            {"topic": "Python"}
        )
    )
)

"Sure, here's a classic one for you:\n\n**Why do Python programmers prefer using the *snake* case?**  \nBecause they don’t like to *indent* their jokes! 🐍😄"

## Runnables

Runnables can be 
- executed
    - invoke(), 
    - batch() 
    - and stream()
- inspected,
- and composed

In [ ]:
runnables = [prompt, llm, parser]

In [ ]:
print(f"{repr(prompt).split('(')[0]}")
print(f"{repr(prompt)}")
print(f"{repr(prompt.invoke)}")
print(f"{repr(prompt.batch)}")
print(f"{repr(prompt.stream)}")
# print(f"{repr(llm)}")
# print(f"{repr(parser)}")

PromptTemplate
PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')
<bound method BasePromptTemplate.invoke of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')>
<bound method Runnable.batch of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')>
<bound method Runnable.stream of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')>


**Execute methods**

In [ ]:
for runnable in runnables:
    print(f"{repr(runnable).split('(')[0]}")
    print(f"\tINVOKE: {repr(runnable.invoke)}")
    print(f"\tBATCH: {repr(runnable.batch)}")
    print(f"\tSTREAM: {repr(runnable.stream)}\n")

PromptTemplate
	INVOKE: <bound method BasePromptTemplate.invoke of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')>
	BATCH: <bound method Runnable.batch of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')>
	STREAM: <bound method Runnable.stream of PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')>

ChatOpenAI
	INVOKE: <bound method BaseChatModel.invoke of ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7aed22f71940>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7aed22a24c80>, root_client=<openai.OpenAI object at 0x7aed230e69f0>, root_async_client=<openai.AsyncOpenAI object at 0x7aed22f18380>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('*****

**Inspect**

In [ ]:
for runnable in runnables:
    print(f"{repr(runnable).split('(')[0]}")
    print(f"\tINPUT: {repr(runnable.get_input_schema())}")
    print(f"\tOUTPUT: {repr(runnable.get_output_schema())}")
    print(f"\tCONFIG: {repr(runnable.config_schema())}\n")

PromptTemplate
	INPUT: <class 'langchain_core.utils.pydantic.PromptInput'>
	OUTPUT: <class 'langchain_core.prompts.prompt.PromptTemplateOutput'>
	CONFIG: <class 'langchain_core.utils.pydantic.PromptTemplateConfig'>

ChatOpenAI
	INPUT: <class 'langchain_openai.chat_models.base.ChatOpenAIInput'>
	OUTPUT: <class 'langchain_openai.chat_models.base.ChatOpenAIOutput'>
	CONFIG: <class 'langchain_core.utils.pydantic.ChatOpenAIConfig'>

StrOutputParser
	INPUT: <class 'langchain_core.output_parsers.string.StrOutputParserInput'>
	OUTPUT: <class 'langchain_core.output_parsers.string.StrOutputParserOutput'>
	CONFIG: <class 'langchain_core.utils.pydantic.StrOutputParserConfig'>



**Config**

In [ ]:
with collect_runs() as run_collection:
    result = llm.invoke(
        "Hello", 
        config={
            'run_name': 'demo_run', 
            'tags': ['demo', 'lcel'], 
            'metadata': {'lesson': 2}
        }
    )

In [ ]:
run_collection.traced_runs

[RunTree(id=721e8109-617f-43a5-bde4-2182c0ecb3e1, name='demo_run', run_type='llm', dotted_order='20250314T200800260294Z721e8109-617f-43a5-bde4-2182c0ecb3e1')]

In [ ]:
run_collection.traced_runs[0].model_dump()

{'id': UUID('019d306d-9a34-7a42-a939-f477bc3384ac'),
 'name': 'demo_run',
 'start_time': datetime.datetime(2026, 3, 27, 17, 53, 2, 523396, tzinfo=datetime.timezone.utc),
 'run_type': 'llm',
 'end_time': datetime.datetime(2026, 3, 27, 17, 53, 4, 75304, tzinfo=datetime.timezone.utc),
 'extra': {'invocation_params': {'model': 'openai/gpt-oss-120b',
   'model_name': 'openai/gpt-oss-120b',
   'stream': False,
   'temperature': 0.7,
   '_type': 'openai-chat',
   'stop': None},
  'options': {'stop': None},
  'batch_size': 1,
  'metadata': {'model': 'openai/gpt-oss-120b',
   'model_name': 'openai/gpt-oss-120b',
   'stream': False,
   'temperature': 0.7,
   '_type': 'openai-chat',
   'lesson': 2,
   'ls_provider': 'openai',
   'ls_model_name': 'openai/gpt-oss-120b',
   'ls_model_type': 'chat',
   'ls_temperature': 0.7,
   'ls_integration': 'langchain_chat_model'}},
 'error': None,
 'serialized': {'lc': 1,
  'type': 'constructor',
  'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'],
  '

**Compose Runnables**

In [ ]:
chain = RunnableSequence(prompt, llm, parser)

In [ ]:
type(chain)

langchain_core.runnables.base.RunnableSequence

In [ ]:
chain.invoke({"topic": "Python"})

'Sure, here’s a Python‑programmer‑friendly one:\n\n```python\n>>> import antigravity\n```\n\nWhy did the Pythonista bring a ladder to the code review?\n\nBecause they heard the project was full of **high‑level** abstractions! 🐍😄'

In [ ]:
for chunk in chain.stream({"topic": "Python"}):
    print(chunk, end="", flush=True)

Sure, here's a Python‑themed chuckle for you:

Why do Python programmers prefer using **list comprehensions**?

Because they’re *list*‑tly obsessed with making everything **compact** and **readable**—they can’t *handle* the extra whitespace! 🐍💻

In [ ]:
chain.batch([
    {"topic": "Python"},
    {"topic": "Data"},
    {"topic": "Machine Learning"},
])

["Sure, here's a Python‑themed one for you:\n\n> Why do Python programmers prefer dark mode?  \n>  \n> Because they don't want to deal with *byte*‑sized bugs! 🐍💡",
 'Sure, here’s one for the data‑loving crowd:\n\n> Why did the dataset break up with the spreadsheet?  \n> Because it found out the spreadsheet was *cell*-fish and kept *pivot*-ing all the relationships!',
 "Sure, here's one for you:\n\n*Why did the neural network go to therapy?*\n\n*Because it had too many hidden layers and couldn't express its true **bias**!*"]

In [ ]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


**Turn any function into a runnable**

In [ ]:
def double(x:int)->int:
    return 2*x

In [ ]:
runnable = RunnableLambda(double)
runnable.invoke(2)

4

**Parallel Runnables**

In [ ]:
parallel_chain = RunnableParallel(
    double=RunnableLambda(lambda x: x * 2),
    triple=RunnableLambda(lambda x: x * 3),
)

In [ ]:
parallel_chain.invoke(3)

{'double': 6, 'triple': 9}

In [ ]:
parallel_chain.get_graph().print_ascii()

+------------------------------+   
| Parallel<double,triple>Input |   
+------------------------------+   
           **        **            
         **            **          
        *                *         
  +--------+          +--------+   
  | Lambda |          | Lambda |   
  +--------+          +--------+   
           **        **            
             **    **              
               *  *                
+-------------------------------+  
| Parallel<double,triple>Output |  
+-------------------------------+  


## LCEL

In [ ]:
prompt

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')

In [ ]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x12025b790>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11559cdd0>, root_client=<openai.OpenAI object at 0x117ed64d0>, root_async_client=<openai.AsyncOpenAI object at 0x120471710>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [ ]:
parser

StrOutputParser()

In [ ]:
chain = RunnableSequence(prompt, llm, parser)
chain

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x12025b790>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11559cdd0>, root_client=<openai.OpenAI object at 0x117ed64d0>, root_async_client=<openai.AsyncOpenAI object at 0x120471710>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [ ]:
prompt | llm | parser

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x12025b790>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11559cdd0>, root_client=<openai.OpenAI object at 0x117ed64d0>, root_async_client=<openai.AsyncOpenAI object at 0x120471710>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [ ]:
chain = prompt | llm | parser

In [ ]:
chain.invoke(
    {"topic": "computer"}
)

'Why did the computer go to therapy?\n\nBecause it had too many bytes from its past!'